Pipeline:
    
    1. SFT teach how to answer
    2. Reward Model teach judge what is good
    3. PPO-RLHF practice with reward
    4. Evaluation + regression tests confirm real improvement

=> A base model learns **"how text usually continues on the internet"**; an assistant must learn **"how to respond responsibly to a user request."**

=> RLHF is not a training script. it is an artifact pipeline. Each stage must leave behind data, models, metrics, and failure cases.

# Reward Model

In [3]:
import torch
import torch.nn as nn

In [55]:
class RewardModel(nn.Module):
    def __init__(self,base_model,hidden_dim=1024):
        super().__init__()
        self.base_model = base_model
        self.reward_head = nn.Linear(hidden_dim,1,dtype=base_model.dtype)
    def forward(self,input_ids):
        response_hs = self.base_model(input_ids,output_hidden_states=True).hidden_states[-1][:,-1] # [B,1]
        return self.reward_head(response_hs) # [,B]

In [53]:
def bradely_terry_loss(rm,chosen_ids,rejected_ids):
    chosen_reward = rm(chosen_ids)
    rejected_reward = rm(rejected_ids)
    loss = - torch.nn.functional.logsigmoid(chosen_reward - rejected_reward).mean()
    return loss

In [13]:
from transformers import AutoTokenizer,AutoModelForCausalLM
model_ids = "LiquidAI/LFM2.5-230M"
tokenizer = AutoTokenizer.from_pretrained(model_ids)
model = AutoModelForCausalLM.from_pretrained(model_ids)

Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]

In [62]:
prompt = "what is 1+1? "
chosen = "the answer is 2"
rejected = "who knows"

In [63]:
chosen_ids = tokenizer.encode(prompt+chosen,return_tensors="pt")
rejected_ids = tokenizer.encode(prompt+rejected,return_tensors="pt")

In [64]:
rm = RewardModel(model)
rm(chosen_ids),rm(rejected_ids)

(tensor([[-0.5703]], dtype=torch.bfloat16, grad_fn=<AddmmBackward0>),
 tensor([[-0.9609]], dtype=torch.bfloat16, grad_fn=<AddmmBackward0>))

In [65]:
bradely_terry_loss(rm,chosen_ids,rejected_ids)

tensor(0.5156, dtype=torch.bfloat16, grad_fn=<NegBackward0>)

In [68]:
rm_config = {
    # Base model: typically a smaller post-SFT model
    "base_model": "sft_model_3b",

    # Learning rate: more conservative than SFT
    "learning_rate": 5e-6,  # SFT typically uses 1e-5 to 2e-5

    # LR schedule: linear warmup + cosine decay
    "warmup_steps": 100,
    "lr_scheduler": "cosine",

    # Gradient clipping: prevent gradient explosion
    "max_grad_norm": 1.0,

    # Batch size: number of preference pairs
    "batch_size": 128,  # 128 (chosen, rejected) pairs per batch

    # Training epochs: usually only 1–2 epochs
    "epochs": 1,  # RM is prone to overfitting; do not train too many epochs
}

# Combined Reward

In [66]:
def format_reward(response:str)->float:
    return 0.5
def check_answer_correct(prompt:str,response:str)->float:
    return 0.4
def length_reward(response:str)->float:
    return -0.01*max(0,len(response)-500)

In [67]:
def combined_reward(rm,prompt,response,reasoning):
    r_rm = rm(prompt,response)
    r_format = format_reward(response)
    r_correct = check_answer_correct(prompt,response)
    r_length = length_reward(response)
    return r_rm + r_format + r_correct + r_length

# Generate Preference Dataset using RLAIF

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "LiquidAI/LFM2.5-230M"
model = AutoModelForCausalLM.from_pretrained(model_id,device_map="mps")
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]

In [2]:
# prompt to generate preference dataset
prompts = [
    "Every day, a tree drops 7 leaves. How many leaves would it drop in a month of February in a non-leap year? Include your logic.",
    "Write a poem about coffee in the style of Emily Dickinson.",
    "Do you know any jokes about animals with ailments?",
    "Compare and contrast the tea ceremonies of two different cultures, discussing their historical origins, cultural significance, and distinct ceremonial elements."
]

In [7]:
prompts_templates = [tokenizer.apply_chat_template([{"role":"user","content":prompt}],tokenize=False,add_generation_prompt=True) for prompt in prompts]
prompts_templates[0]

'<|startoftext|><|im_start|>user\nEvery day, a tree drops 7 leaves. How many leaves would it drop in a month of February in a non-leap year? Include your logic.<|im_end|>\n<|im_start|>assistant\n'

In [41]:
def generate_answer(prompt,model,tokenizer,k=1):
    input_ids = tokenizer(prompt,return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **input_ids,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.8,
        top_p = 0.7,
        num_return_sequences=k
    )
    prefix_ids = output_ids[:,input_ids["input_ids"].shape[-1]:]
    response = tokenizer.batch_decode(prefix_ids,skip_special_tokens=True)
    return response

In [42]:
generate_answer(prompts_templates[0],model,tokenizer,k=2)

['To determine how many leaves a tree would drop in a month of February in a non-leap year, we need to calculate the number of leaves per day and then multiply that by 30 (the number of days in a month).\n\n1. **Calculate the number of leaves per day:**\n   \\[\n   \\text{Leaves per day} = 7\n   \\]\n\n2. **Calculate the number of leaves in a month:**\n   \\[\n   \\text{Leaves in a month} = 7 \\, \\text{leaves/day} \\times 30 \\, \\text{days/month} = 210 \\, \\text{leaves}\n   \\]\n\nThus, in a month of February in a non-leap year, the tree would drop \\(\\boxed{210}\\) leaves.',
 'To determine how many leaves a tree drops each day in a month of February, we first need to calculate the number of leaves dropped per day and then multiply by 30 (since there are 30 days in a month).\n\n1. **Calculate the number of leaves dropped per day:**\n   \\[\n   \\text{Leaves per day} = 7\n   \\]\n\n2. **Calculate the number of leaves dropped in a month:**\n   \\[\n   \\text{Leaves in a month} = 7 \\

In [44]:
# generate k response for each prompt
from tqdm import tqdm
K = 3
prompt_response_ds = {"prompts":[],"responses":[]}
for prompt in tqdm(prompts_templates):
    prompt_response_ds["prompts"].append(prompt)
    responses = generate_answer(prompt,model,tokenizer,k=3)
    prompt_response_ds["responses"].append(responses)

100%|██████████| 4/4 [03:34<00:00, 53.73s/it]


In [45]:
import pandas as pd
pd.DataFrame(prompt_response_ds)

,prompts,responses
0,"<|startoftext|><|im_start|>user\nEvery day, a ...",[To determine how many leaves the tree drops i...
1,<|startoftext|><|im_start|>user\nWrite a poem ...,"[In the hush between dawn's first light, \nA ..."
2,<|startoftext|><|im_start|>user\nDo you know a...,[I don’t have personal experiences or animal-r...
3,<|startoftext|><|im_start|>user\nCompare and c...,[Certainly! Let's compare and contrast the **J...


In [46]:
from dotenv import load_dotenv
import os
from openai import OpenAI
load_dotenv()
client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = os.environ.get("GROQ_API_KEY")
)
# test
client.chat.completions.create(model="openai/gpt-oss-120b", messages=[{"role":"user","content":"hello!"}])

ChatCompletion(id='chatcmpl-cd09d1a1-f454-4f8a-ba18-7475b9c2f0b8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I help you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user says "hello!". We just respond friendly.'))], created=1784476071, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_c800245357', usage=CompletionUsage(completion_tokens=30, prompt_tokens=73, total_tokens=103, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=12, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.017573179, prompt_time=0.002796256, completion_time=0.063419516, total_time=0.066215772), usage_breakdown=None, x_groq={'id': 'req_01kxxgzxphec78vpmf1xxbq395', 'seed': 1609608985})

In [47]:
rlaif_judge_prompt = """
You are a strict answer quality evaluator. Compare two responses.

Evaluation dimensions:
1. Accuracy: Are the facts correct, any hallucinations?
2. Helpfulness: Does it genuinely address the user's question?
3. Clarity: Is the writing clear and the logic coherent?
4. Safety: Does it contain harmful, biased, or misleading content?

User question:
{prompt}

Response A:
{response_a}

Response B:
{response_b}

Output only JSON:
{{"winner": "A" or "B" or "tie", "reason": "one-sentence reason"}}
"""

In [63]:
import json
preferences_ = {"prompt":[],"chosen":[],"rejected":[],"judge":[]}
for pi,prompt in enumerate(prompts):
    for i in range(K-1):
        for j in range(i+1,K):
            response_a = prompt_response_ds["responses"][pi][i]
            response_b = prompt_response_ds["responses"][pi][j]
            finall_prompt = rlaif_judge_prompt.format(prompt=prompt,response_a=response_a,response_b=response_b)
            client_response = client.chat.completions.create(model="openai/gpt-oss-120b", messages=[{"role":"user","content":finall_prompt}])
            response = client_response.choices[0].message.content
            try:
                response = json.loads(response)
                preferences_["judge"].append(response)
                preferences_["prompt"].append(prompt)

                if response["winner"] == "A" or response["winner"]=="tie":
                    preferences_["chosen"].append(response_a)
                    preferences_["rejected"].append(response_b)
                else:
                    preferences_["chosen"].append(response_b)
                    preferences_["rejected"].append(response_a)
            except:
                continue


In [64]:
pd.DataFrame(preferences_)

,prompt,chosen,rejected,judge
0,"Every day, a tree drops 7 leaves. How many lea...",To determine how many leaves the tree drops in...,To determine how many leaves a tree drops in a...,"{'winner': 'tie', 'reason': 'Both answers cont..."
1,"Every day, a tree drops 7 leaves. How many lea...",To determine how many leaves a tree would drop...,To determine how many leaves the tree drops in...,"{'winner': 'B', 'reason': 'Response B gives th..."
2,"Every day, a tree drops 7 leaves. How many lea...",To determine how many leaves a tree would drop...,To determine how many leaves a tree drops in a...,"{'winner': 'B', 'reason': 'Response B gives th..."
3,Write a poem about coffee in the style of Emil...,"In the hush between dawn's first light, \nA q...","In the quiet hush of dawn's first light, \nA ...","{'winner': 'A', 'reason': 'Both poems are safe..."
4,Write a poem about coffee in the style of Emil...,"In the hush between dawn's first light, \nA q...","In quiet corners where shadows play, \nA cup ...","{'winner': 'A', 'reason': 'Response A stays on..."
5,Write a poem about coffee in the style of Emil...,"In the quiet hush of dawn's first light, \nA ...","In quiet corners where shadows play, \nA cup ...","{'winner': 'A', 'reason': 'Response A correctl..."
6,Do you know any jokes about animals with ailme...,I don’t have personal experiences or animal-re...,I don’t have personal experiences or animal-re...,"{'winner': 'A', 'reason': 'Response A is sligh..."
7,Do you know any jokes about animals with ailme...,"I'm sorry, but I can't share jokes or content ...",I don’t have personal experiences or animal-re...,"{'winner': 'B', 'reason': 'B safely declines t..."
8,Do you know any jokes about animals with ailme...,"I'm sorry, but I can't share jokes or content ...",I don’t have personal experiences or animal-re...,"{'winner': 'B', 'reason': 'Response B safely d..."
9,Compare and contrast the tea ceremonies of two...,Certainly! Let's compare and contrast the **Ja...,Certainly! Let's compare and contrast the **Ja...,"{'winner': 'B', 'reason': 'Response B provides..."


In [ ]:
# 